In [ ]:
##                                                       ##
##                This cell is not used                  ##
##                                                       ##

#run the simulation of the diffusion planner on the minisplit  
!python /home/taimor/V0.1-diffusion-es/nuplan-devkit/nuplan/planning/script/run_simulation.py \
  experiment_name=dp_mini_sanity_2 \
  scenario_builder=nuplan_mini \
  scenario_builder.data_root=/home/taimor/data1/nuplan-v1.1/splits/mini \
  +simulation=closed_loop_nonreactive_agents \
  planner=diffusion_planner \
  planner.diffusion_planner.config.args_file=/home/taimor/V0.1-diffusion-es/Diffusion-Planner/checkpoints/args.json \
  planner.diffusion_planner.ckpt_path=/home/taimor/V0.1-diffusion-es/Diffusion-Planner/checkpoints/model.pth \
  scenario_filter.shuffle=true scenario_filter.limit_total_scenarios=1 \
  worker=sequential verbose=true \
  'hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter,pkg://diffusion_planner.config,pkg://nuplan.planning.script.config.common,pkg://nuplan.planning.script.experiments]'



In [ ]:
##                                                       ##
##                 visualize in nuboard                  ##
##                                                       ##

!python /home/shadi/Project-diffusion/nuplan-devkit/nuplan/planning/script/run_nuboard.py \
  port_number=7007 \
  worker=sequential \
  simulation_path='["/home/shadi/nuplan/exp/exp/dp_mini_sanity_3/closed_loop_nonreactive_agents/"]' \
  scenario_builder.data_root=/home/shadi/nuplan/data/cache/mini \
  scenario_builder.map_root=/home/shadi/nuplan/maps \



In [ ]:
##                                                       ##
##                Fetching Scenario Info                 ##
##            (nuplan_mini, log ...01486 only)           ##
##                                                       ##

import os
from nuplan.planning.scenario_builder.nuplan_db.nuplan_scenario_builder import NuPlanScenarioBuilder
from nuplan.planning.scenario_builder.scenario_filter import ScenarioFilter
from nuplan.planning.utils.multithreading.worker_parallel import SingleMachineParallelExecutor

DATA_ROOT   = "/home/shadi/nuplan/data/cache/mini"   # same as in sim
MAP_ROOT    = os.environ["NUPLAN_MAPS_ROOT"]
SENSOR_ROOT = DATA_ROOT
MAP_VERSION = "nuplan-maps-v1.0"

# Path to the specific mini log you care about
LOG_FILE = os.path.join(
    DATA_ROOT,
    "2021.07.16.20.45.29_veh-35_01095_01486.db",
)

builder = NuPlanScenarioBuilder(
    data_root=DATA_ROOT,
    map_root=MAP_ROOT,
    sensor_root=SENSOR_ROOT,
    db_files=[LOG_FILE],         # <<< ONLY this .db file
    map_version=MAP_VERSION,
    include_cameras=False,
    max_workers=4,
    verbose=True,
)

# Match nuplan_challenge_scenarios.yaml
challenge_types = [
    "starting_left_turn",
    "starting_right_turn",
    "starting_straight_traffic_light_intersection_traversal",
    "stopping_with_lead",
    "high_lateral_acceleration",
    "high_magnitude_speed",
    "low_magnitude_speed",
    "traversing_pickup_dropoff",
    "waiting_for_pedestrian_to_cross",
    "behind_long_vehicle",
    "stationary_in_traffic",
    "near_multiple_vehicles",
    "changing_lane",
    "following_lane_with_lead",
]

scenario_filter = ScenarioFilter(
    scenario_types=challenge_types,
    scenario_tokens=None,
    log_names=None, #["2021.07.16.20.45.29_veh-35_01095_01486"],  # base log name
    map_names=None,
    num_scenarios_per_type=None,
    limit_total_scenarios=None,   # adjust if needed
    timestamp_threshold_s=None,
    ego_displacement_minimum_m=None,
    ego_start_speed_threshold=None,
    ego_stop_speed_threshold=None,
    speed_noise_tolerance=None,
    expand_scenarios=False,
    remove_invalid_goals=True,
    shuffle=False,
)

worker = SingleMachineParallelExecutor(use_process_pool=False, max_workers=4)
scenarios = builder.get_scenarios(scenario_filter, worker)

for i, s in enumerate(scenarios):
    print(f"{i:03d} token={s.token}  type={s.scenario_type}  log={s.log_name}")


In [ ]:
##                                                       ##
## This cell is for running the simulation in debug mode ##
##                                                       ##

import sys
import os
os.environ["HYDRA_FULL_ERROR"] = "1"  # Enable full Hydra error messages

# --- 0. Patch asyncio for Jupyter ---
# This is required to fix "RuntimeError: asyncio.run() cannot be called from a running event loop"
# because Jupyter/IPython already runs its own event loop.
import nest_asyncio
nest_asyncio.apply()
# ------------------------------------

# --- 1. Set up Python paths ---
# Add the project subdirectories to the Python path.
# Assumes you are running this notebook from '/home/taimor/V0.1-diffusion-es/'
devkit_path = 'nuplan-devkit'
if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)

# Add the Diffusion-Planner project root to the Python path
# so that hydra's `pkg://diffusion_planner` resolver can find it.
planner_path = 'Diffusion-Planner'
if planner_path not in sys.path:
    sys.path.insert(0, planner_path)

print(f"Updated sys.path:\n{sys.path}\n")

try:
    # --- 2. Import the main function ---
    # Now that the path is set, we can import the main function from the script.
    # The `@hydra.main` decorator is already applied.
    from nuplan.planning.script.run_simulation import main as run_simulation_main

    # --- 3. Define the arguments ---
    # This list mimics your shell command.
    # The first item is the script name, which hydra expects.
    script_path = 'nuplan-devkit/nuplan/planning/script/run_simulation.py'

    #=================================================================================#
    #logic to be able to break line of large number of tokens, and append later to args
    #=================================================================================#

        # type = high_magnitude_speed   : modified score > original score
        # "scenario_filter.scenario_tokens=[\"0a13cbf7df6959d7\", \"1a30cfb28c235543\", \"2a7f8e03e9205459\"]",
        # type = near_multiple_vehicles : modified score < original score
        # "scenario_filter.scenario_tokens=[\"0390ae0fac3953d1\", \"131996f9cd3d5343\", \"732e2a4d42135cec\"]",
        # type = 10 multiple types      : modified score > original score
        # "scenario_filter.scenario_tokens=[\"ffa83e1b737a5975\", \"ff1dd74e8d075989\", \"ff00d647b4645efb\", \"ff006a67f6bf571b\", \"fe81d22a27b95cf7\", \"f68f4658921d5078\", \"f6731b531da9504f\", \"f118ccba1fa45f93\", \"f040b0da10e35da6\", \"ef3fa40447fc56dc\"]",
        # type = 11 multiple types      : modified score ? original score
        # "scenario_filter.scenario_tokens=[\"526840f14e8b5a3f\", \"78b4b5dbbbe25c40\", \"a64086ea795f5df1\", \"e55cc719e4405b03\", \"52efcecc45955a2a\", \"96bb1dcef5be5299\", \"f542c53419545043\", \"6bb3d94d109159d9\", \"e073c39ce158544b\", \"279431ebb0e250df\", \"552051da1d505a0f\"]",
        # type = 9 high_magnitude_speed : modified score ? original score
        # "scenario_filter.scenario_tokens=[\"1453faa7a81b5dd4\", \"3445491a26c156c1\", \"54567345bab15127\", \"644602ae738c5c2d\", \"741416c1569c5337\", \"942b9df7477e5609\", \"b4a2d642b2cf502b\", \"d46d2283d1c65c94\", \"f45f81ef04ec5806\"]",
        # TOKENS = [
        #     "0a59682e2631549d", "13386f837bcd5d52","1d68d66ecb9a5a83", "27144eb726ad5364", "321b5583f19754c9",
        #     "3f65113d0f4852bc", "473df59892b45a2c","504288eec3f25c7a", "5c73984548345bda", "65dc13856fe95899",
        #     "6e2f09a629d05e05", "7b7da8f5908a5366","8869ff3c518c5177", "90f7299c57c25290", "9befb7a1a414560e",
        #     "a64086ea795f5df1", "b0a7cd3751065ec5","bbbc9f99604250cc", "c8d3d51ea6b45d7b", "d577e813f668517e"
        # ]
    TOKENS = [
        "ffa83e1b737a5975", "ff1dd74e8d075989", "ff00d647b4645efb", "ff006a67f6bf571b", "fe81d22a27b95cf7",
        "f68f4658921d5078", "f6731b531da9504f", "f118ccba1fa45f93", "f040b0da10e35da6", "ef3fa40447fc56dc"
    ]
    scenario_tokens_override = (
        "scenario_filter.scenario_tokens=["
        + ", ".join(f"\"{t}\"" for t in TOKENS)
        + "]"
    )
    
    args = [
        script_path,  # sys.argv[0]
        "experiment_name=dp_mini_sanity_3",
        "scenario_builder=nuplan_mini",
        "scenario_builder.data_root=/home/shadi/nuplan/data/cache/mini", # Kept as absolute path
        "+simulation=closed_loop_nonreactive_agents",
        "planner=diffusion_planner",
        "planner.diffusion_planner.config.args_file=/home/shadi/Project-diffusion/Diffusion-Planner/checkpoints/args.json",
        "planner.diffusion_planner.ckpt_path=/home/shadi/Project-diffusion/Diffusion-Planner/checkpoints/model.pth",
        "scenario_filter.shuffle=false",
        ###########################################
        "scenario_filter.limit_total_scenarios=10",
        ###########################################
        "worker=sequential",
        "verbose=true",
        "hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter,pkg://diffusion_planner.config,pkg://nuplan.planning.script.config.common,pkg://nuplan.planning.script.experiments]"
    ]

    #CONTINUATION: logic to be able to break line of large number of tokens
    args.append(scenario_tokens_override)



    # --- 4. Run the simulation ---
    print("Backing up original sys.argv...")
    original_argv = list(sys.argv)  # Make a copy
    
    try:
        print("Setting new sys.argv for hydra...")
        sys.argv = args
        print(f"Running simulation with args: {sys.argv}")
        
        # Call the hydra-decorated main function
        run_simulation_main()
        
        print("\nSimulation finished.")
        
    except Exception as e:
        print(f"\nAn error occurred during simulation: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        # Restore the original sys.argv so it doesn't affect other notebook cells
        print("Restoring original sys.argv...")
        sys.argv = original_argv

except ImportError as e:
    print(f"Error: Failed to import modules: {e}")
    print("Please double-check the paths set in this cell:")
    print(f"Devkit path: {devkit_path}")
    print(f"Planner path: {planner_path}")
    print("Ensure these directories exist and contain the correct packages.")


